# Polling Station Finder — Design Journey & Open Issues

## Development Phases

### Phase 1 — MVP Search
Built a simple address-lookup tool: a single search box using **token-based AND matching** against the `full_address` column. Debounced HTMX input (`keyup changed delay:300ms`) sends queries as the user types, returning up to 20 clickable results. Clicking a result fetches the polling station name and a **Google Maps directions** link. No page reloads — pure HTMX swap into a `#results` div.

### Phase 2 — Map Display
Added a **Leaflet** map (stacked below the search bar for mobile readiness) showing all **116 polling district boundaries** loaded from a GeoJSON file. Districts drawn with light orange fill (`#fed7aa`) and grey borders. The GeoJSON is served from memory via a `/districts.json` route with a 24-hour browser cache header. OpenStreetMap tiles used as the base layer.

### Phase 3 — Map Interactivity
Implemented the **HX-Trigger** pattern: when a user selects an address, the `/station/{uprn}` response includes an HTTP header carrying the district code and station coordinates. A client-side JS listener catches this event and:
- **Highlights** the selected district (blue border + lighter fill)
- **Drops a marker** at the polling station with a popup showing the station name
- **Zooms** the map to fit the selected district
- A **Reset** button (top-right) clears the search, removes the marker, and resets all district styles

## Open Issues / Technical Debt

1. **External `map.js` loading** — using `defer` caused timing issues with Leaflet CDN load order; inline JS used as a workaround. Fix: wrap in `window.addEventListener('load', ...)`.
2. **Double-init guard** — `window._pollingMapInit` flag prevents re-initialisation but is fragile if HTMX swaps the whole page.
3. **No error handling** on `/station/{uprn}` — if a UPRN is not found, the user sees an unhandled error instead of a friendly message.
4. **GeoJSON served from memory** — fast, but requires a server restart if the data file changes.
5. **Polygon alignment** — district boundaries appear shifted east and do not match the correct borough boundaries. Likely a CRS/datum issue in the source GeoJSON export (e.g. residual BNG artefact). Needs investigation: compare a known landmark's coordinates against the polygon vertices to determine if it's a uniform offset.
6. **Mobile map height** — fixed at `600px`; should consider responsive `vh` units for varying screen sizes.

# 1. Setup & Imports

In [ ]:
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
import fasthtml.components as fc
from starlette.staticfiles import StaticFiles
from starlette.responses import JSONResponse, HTMLResponse
import subprocess, json
import pandas as pd
from fastlite import database
from functools import partial

# Kill anything on port 8000 to avoid "address already in use"
def kill_port(port=8000):
    subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

kill_port()

# ── Headers: DaisyUI + Tailwind + Leaflet ──
hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
    Link(href='https://unpkg.com/leaflet@1.9.4/dist/leaflet.css', rel='stylesheet'),
    Script(src='https://unpkg.com/leaflet@1.9.4/dist/leaflet.js'),
)

/usr/local/lib/python3.12/site-packages/fasthtml/jupyter.py:136: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient


# 2. Database

Two normalised tables built from `addresses2.csv`:

- **`addresses`** — one row per UPRN, linked to its polling station via `station_uprn`.
- **`stations`** — one row per unique `Polling_Station_UPRN`, deduplicated from the CSV.

Both tables store **WGS84 latitude / longitude** (converted from British National Grid with `pyproj` EPSG:27700 → EPSG:4326). All text columns are cleaned of `_x000D_` Excel artifacts and stray whitespace before insert.

Set `INSERT_DB = True` to rebuild the database from CSV.

In [ ]:
from pyproj import Transformer

# ── Configuration ──
INSERT_DB = False   # Set True to rebuild database from CSV

db = database('polling.db')

# ── 1. Clean slate (only if inserting) ──
if INSERT_DB:
    if 'addresses_tbl' in globals(): addresses_tbl.drop(ignore=True)
    if 'stations_tbl'  in globals(): stations_tbl.drop(ignore=True)

# ── 2. Create tables ──
addresses_tbl = db.t.addresses.create(
    uprn=int, full_address=str, street=str, town=str, postcode=str,
    address_line1=str, address_line2=str, blpu_class=str,
    polling_district=str, station_uprn=int,
    easting=float, northing=float, latitude=float, longitude=float,
    pk='uprn', if_not_exists=True)

stations_tbl = db.t.stations.create(
    uprn=int, name=str,
    easting=float, northing=float, latitude=float, longitude=float,
    pk='uprn', if_not_exists=True)

# ── 3. Load & insert (only when INSERT_DB is True) ──
if INSERT_DB:
    df = (pd.read_csv('addresses2.csv', index_col=0)
            .dropna(subset=['Polling_Station_UPRN'])
            .drop_duplicates(keep='first'))
    df['Polling_Station_UPRN'] = df['Polling_Station_UPRN'].astype(int)

    # Clean text columns
    def clean_text(s):
        return (s.str.replace('_x000D_', ' ', regex=False)
                  .str.replace('\r', '').str.replace('\n', ' ')
                  .str.strip())

    text_cols = ['POLLING_PLACE_NAME','FULLADDRESS','STREET','TOWN',
                 'POSTCODE','ADDRESSLINE1','ADDRESSLINE2','BLPUCLASS',
                 'POLLING_DISTRICT']
    df[text_cols] = df[text_cols].apply(clean_text)

    # Coordinate transformer (BNG → WGS84)
    transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
    def en_to_ll(e, n):
        lon, lat = transformer.transform(e, n)
        return float(lat), float(lon)

    # Build stations DataFrame
    stations = (df[['Polling_Station_UPRN','POLLING_PLACE_NAME','EASTING','NORTHING']]
                .drop_duplicates('Polling_Station_UPRN')
                .rename(columns=dict(
                    Polling_Station_UPRN='uprn', POLLING_PLACE_NAME='name',
                    EASTING='easting', NORTHING='northing')))
    stations[['latitude','longitude']] = stations.apply(
        lambda r: pd.Series(en_to_ll(r.easting, r.northing)), axis=1)

    # Build addresses DataFrame
    addr_cols = dict(
        UPRN='uprn', FULLADDRESS='full_address', STREET='street',
        TOWN='town', POSTCODE='postcode', ADDRESSLINE1='address_line1',
        ADDRESSLINE2='address_line2', BLPUCLASS='blpu_class',
        POLLING_DISTRICT='polling_district',
        Polling_Station_UPRN='station_uprn', X='easting', Y='northing')
    addrs = df.rename(columns=addr_cols)[list(addr_cols.values())]
    addrs[['full_address','street','town','postcode',
           'address_line1','address_line2','blpu_class','polling_district']] = \
        addrs[['full_address','street','town','postcode',
               'address_line1','address_line2','blpu_class','polling_district']].fillna('')
    addrs[['latitude','longitude']] = addrs.apply(
        lambda r: pd.Series(en_to_ll(r.easting, r.northing)), axis=1)

    # Persist
    stations_tbl.insert_all(stations.to_dict(orient='records'))
    addresses_tbl.insert_all(addrs.to_dict(orient='records'))
    print(f"Inserted {len(stations)} stations, {len(addrs)} addresses")
else: print(f"Using existing DB: {addresses_tbl.count} addresses, {stations_tbl.count} stations")

Using existing DB: 128873 addresses, 85 stations


# 3. App & GeoJSON

In [ ]:
# ── App + server ──
if 'srv' not in globals() or not srv:
    app = FastHTML(hdrs=hdrs, session_cookie='polling_session')
    app.mount('/static', StaticFiles(directory='static'), name='static')
    rt = app.route
    srv = JupyUvi(app)

def get_preview(app): return partial(HTMX, app=app, host=None, port=None)
preview = get_preview(app)

# ── Load GeoJSON once into memory ──
gj = json.load(open('PollingDistricts260428.json'))
print(f"Loaded {len(gj['features'])} district features")

# ── Serve GeoJSON (cached 24h) ──
@rt('/districts.json')
def districts_json():
    return JSONResponse(gj, headers={"cache-control": "public, max-age=86400"})

Loaded 116 district features


# 4. Search & Station Routes

The search page uses a stacked layout (search box → results → map) for mobile readiness.

- **Token-based AND search**: query is split on whitespace; every token must appear in `full_address` (order-independent)
- **Debounced input**: HTMX fires after 300ms pause, minimum 3 characters
- **HX-Trigger pattern**: `/station/{uprn}` returns station info HTML *and* an `HX-Trigger` header carrying district code + station coordinates, which the client-side map listener uses to highlight the district and drop a marker

In [ ]:
# ── Map JavaScript (inline — see Open Issues #1 for external file notes) ──
MAP_JS = """
(function () {
    if (window._pollingMapInit) return;
    window._pollingMapInit = true;

    const DEFAULT_STYLE = { color: '#666', weight: 1, fillColor: '#fed7aa', fillOpacity: 0.60 };
    const HIGHLIGHT_STYLE = { color: '#3b82f6', weight: 3, fillColor: '#93c5fd', fillOpacity: 0.40 };

    const map = L.map('map').setView([51.45, -0.12], 12);
    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
        attribution: '&copy; OpenStreetMap contributors'
    }).addTo(map);

    let districtsLayer;

    fetch('/districts.json')
        .then(r => r.json())
        .then(gj => {
            districtsLayer = L.geoJSON(gj, { style: DEFAULT_STYLE }).addTo(map);
            map.fitBounds(districtsLayer.getBounds());
        });

    // ── Reset-view control ──
    const ResetControl = L.Control.extend({
        options: { position: 'topright' },
        onAdd: function () {
            const btn = L.DomUtil.create('button', 'leaflet-bar');
            btn.innerHTML = '&#x27F2; Reset';
            btn.style.cssText = 'background:#fff;padding:6px 10px;cursor:pointer;font-size:14px;border:none;';
            btn.title = 'Reset view';
            L.DomEvent.disableClickPropagation(btn);
            btn.onclick = () => {
                if (districtsLayer) {
                    map.fitBounds(districtsLayer.getBounds());
                    districtsLayer.setStyle(DEFAULT_STYLE);
                }
                if (window._stationMarker) { map.removeLayer(window._stationMarker); window._stationMarker = null; }
                const input = document.querySelector('input[name=q]');
                if (input) input.value = '';
                const results = document.getElementById('results');
                if (results) results.innerHTML = '';
            };
            return btn;
        }
    });
    new ResetControl().addTo(map);

    // ── Listen for HX-Trigger from /station/{uprn} ──
    document.addEventListener('showDistrict', function (evt) {
        const data = evt.detail;

        // Highlight selected district, reset others
        let targetLayer = null;
        districtsLayer.eachLayer(function (layer) {
            if (layer.feature.properties.POLLING_DISTRICT === data.district) targetLayer = layer;
        });
        if (targetLayer) {
            districtsLayer.setStyle(DEFAULT_STYLE);
            targetLayer.setStyle(HIGHLIGHT_STYLE);
            map.fitBounds(targetLayer.getBounds());
        }

        // Drop / replace station marker
        if (window._stationMarker) map.removeLayer(window._stationMarker);
        window._stationMarker = L.marker([data.lat, data.lon])
            .addTo(map)
            .bindPopup(data.station_name)
            .openPopup();
    });
})();
"""

# ── Search page ──
@rt
def search_page():
    return Div(cls="px-4 py-6 max-w-7xl mx-auto")(
        H1("Find your polling station", cls="text-2xl font-bold mb-4"),
        Form(
            Input(name="q", placeholder="Type address or postcode...",
                  cls="input input-bordered w-full",
                  hx_post="/search", hx_target="#results", hx_swap="innerHTML",
                  hx_trigger="keyup changed delay:300ms"),
        ),
        Div(id="results", cls="mt-4"),
        Div(id="map", style="height: 600px;", cls="mt-4 rounded-lg"),
        Script(MAP_JS),
    )

# ── Address search (HTMX POST) ──
@rt('/search')
def search_post(q: str = ""):
    tokens = q.split()
    if sum(len(t) for t in tokens) < 3:
        return Div("Type at least 3 characters...", cls="text-sm opacity-60")
    where = " AND ".join(["full_address like ?"] * len(tokens))
    params = [f'%{t}%' for t in tokens]
    rows = addresses_tbl(where, params, limit=20)
    if not rows:
        return Div("No matches found", cls="alert alert-warning")
    return Div(
        *[A(r['full_address'].strip(),
            hx_get=f"/station/{r['uprn']}",
            hx_target="#results", hx_swap="innerHTML",
            cls="block p-2 hover:bg-base-200 cursor-pointer border-b border-base-300")
          for r in rows],
        cls="card bg-base-100 shadow-sm"
    )

# ── Station detail + HX-Trigger for map ──
@rt('/station/{uprn}')
def station_get(uprn: int):
    addr = addresses_tbl[uprn]
    station = stations_tbl[addr['station_uprn']]

    content = Div(
        H2("Your polling station", cls="card-title"),
        P(Strong("Your address: "), addr['full_address']),
        P(Strong("Station: "), station['name']),
        A("Get directions",
          href=f"https://maps.google.com/?q={station['latitude']},{station['longitude']}",
          target="_blank",
          cls="btn bg-orange-500 hover:bg-orange-600 text-white border-none mt-2"),
        cls="card bg-base-100 shadow-sm p-4"
    )

    trigger_data = {"showDistrict": {
        "district": addr['polling_district'],
        "lat": station['latitude'],
        "lon": station['longitude'],
        "station_name": station['name']
    }}

    return HTMLResponse(
        content=to_xml(content),
        headers={"HX-Trigger": json.dumps(trigger_data)}
    )

# 5. Preview & Hosting

This dialog lives in `AUTORUN/website_elections/`, so it runs automatically when the instance restarts.

- **Public URL**: check Dashboard → Instance Settings → Port Mappings for your `*.solveit.pub` URL on port 8000
- **Live route**: `https://<your-public-url>/search_page`